# Лабораторна робота 6. Власна RAG-система: Асистент із документації FastAPI

## 1.1. Корпус

- **Опис корпусу та цільова аудиторія:** База знань на основі офіційної технічної документації Python-фреймворку **FastAPI** (14 ключових розділів: `First Steps`, параметри шляху й запиту, моделі тіла запиту Pydantic, обробка помилок, впровадження залежностей `Depends`, основи безпеки й автентифікація OAuth2/JWT, middleware, CORS, реляційні бази даних SQL/SQLModel, фонові задачі, події життєвого циклу `lifespan` та тестування). Асистент призначений для backend-розробників (зокрема під час розробки REST API-сервісів та дипломування), щоб швидко отримувати точні відповіді щодо параметрів, декораторів, статус-кодів і патернів FastAPI з прямим посиланням на розділ документації.
- **Джерело та ліцензія:** Офіційний репозиторій [`fastapi/fastapi`](https://github.com/fastapi/fastapi/tree/master/docs/en/docs) (ліцензія **MIT License** — дозволяє вільне завантаження, використання та модифікацію).
- **Обсяг:** 14 документів у форматі Markdown (`.md`), загальний обсяг після очищення коментарів — **115 197 символів** (**199** текстових фрагментів).
- **Чому потрібен саме RAG, а не промпт чи донавчання:**
  1. **Чому не промпт:** уся документація FastAPI налічує сотні сторінок (до того ж містить зовнішні приклади коду); вставляти її цілком у кожен запит дорого, повільно й призводить до падіння точності на середині контексту (*Lost in the Middle*).
  2. **Чому не донавчання (SFT):** у FastAPI регулярно змінюються рекомендовані API (наприклад, перехід від застарілого `@app.on_event("startup")` до контекстного менеджера `lifespan` або перехід від `passlib` до `pwdlib` у JWT), а SFT погано запамʼятовує точні сигнатури та не дає верифікованого посилання на конкретний файл документації. RAG дозволяє оновлювати документацію без перетренування моделі та гарантує цитування джерел.


In [1]:
!pip install -q "sentence-transformers>=5" langchain-core langchain-text-splitters langchain-community langchain-huggingface faiss-cpu rank_bm25 gradio

import re
import time
import gradio as gr
import numpy as np
import pandas as pd
import requests
import torch
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Пристрій: {device}")

RAW_BASE = "https://raw.githubusercontent.com/fastapi/fastapi/master/docs/en/docs"
FASTAPI_DOCS = [
    ("first-steps", "First Steps", "tutorial/first-steps.md"),
    ("path-params", "Path Parameters", "tutorial/path-params.md"),
    ("query-params", "Query Parameters", "tutorial/query-params.md"),
    ("body", "Request Body", "tutorial/body.md"),
    ("handling-errors", "Handling Errors", "tutorial/handling-errors.md"),
    ("dependencies", "Dependencies (First Steps)", "tutorial/dependencies/index.md"),
    ("security-first", "Security - First Steps", "tutorial/security/first-steps.md"),
    ("security-jwt", "OAuth2 with Password (and hashing), Bearer with JWT tokens", "tutorial/security/oauth2-jwt.md"),
    ("middleware", "Middleware", "tutorial/middleware.md"),
    ("cors", "CORS (Cross-Origin Resource Sharing)", "tutorial/cors.md"),
    ("sql-databases", "SQL (Relational) Databases", "tutorial/sql-databases.md"),
    ("background-tasks", "Background Tasks", "tutorial/background-tasks.md"),
    ("lifespan", "Lifespan Events", "advanced/events.md"),
    ("testing", "Testing", "tutorial/testing.md"),
]

def clean_markdown(md_text: str) -> str:
    # Прибираємо службові HTML-вставки, коментарі та зайві порожні рядки, зберігаючи заголовки й код
    text = re.sub(r"<!--.*?-->", "", md_text, flags=re.DOTALL)
    text = re.sub(r"\{/\*.*?\*/\}", "", text, flags=re.DOTALL)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

raw_documents = []
total_chars = 0
for doc_id, title, rel_path in FASTAPI_DOCS:
    resp = requests.get(f"{RAW_BASE}/{rel_path}", timeout=30)
    resp.raise_for_status()
    cleaned = clean_markdown(resp.text)
    total_chars += len(cleaned)
    raw_documents.append(Document(page_content=cleaned, metadata={"doc_id": doc_id, "title": title, "path": rel_path}))
    print(f"{doc_id:18s} | {len(cleaned):6d} символів | {title}")

print(f"\nУсього документів: {len(raw_documents)}, загальний обсяг: {total_chars:,} символів")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


/tmp/ipykernel_1019/2385284867.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


Пристрій: cuda
first-steps        |  14057 символів | First Steps
path-params        |   8960 символів | Path Parameters
query-params       |   4848 символів | Query Parameters
body               |   6572 символів | Request Body
handling-errors    |   9107 символів | Handling Errors
dependencies       |   9731 символів | Dependencies (First Steps)
security-first     |   8329 символів | Security - First Steps
security-jwt       |  10725 символів | OAuth2 with Password (and hashing), Bearer with JWT tokens
middleware         |   3927 символів | Middleware
cors               |   5362 символів | CORS (Cross-Origin Resource Sharing)
sql-databases      |  15404 символів | SQL (Relational) Databases
background-tasks   |   4756 символів | Background Tasks
lifespan           |   7933 символів | Lifespan Events
testing            |   5486 символів | Testing

Усього документів: 14, загальний обсяг: 115,197 символів


## 1.2. Бенчмарк

До налаштування параметрів пошуку складено набір із **25 питань** (22 питання з відповідями у корпусі та 3 питання поза корпусом для перевірки режиму відмови). Питання охоплюють різні типи: фактичні (конкретні класи, статус-коди, параметри), процедурні (як налаштувати CORS, JWT, тестування), перефразування без точних назв класів та неможливі питання.

In [2]:
BENCHMARK = [
    # (питання, [очікувані doc_id], regex-доказ у фрагменті, еталонна відповідь)
    ("What HTTP status code does FastAPI return by default when raising HTTPException without specifying details?", ["handling-errors"], r"HTTPException", "HTTPException requires a status_code parameter, while validation errors return 422."),
    ("Which exception class should be raised in FastAPI to return an HTTP error response to the client?", ["handling-errors"], r"HTTPException", "Raise HTTPException from fastapi."),
    ("What HTTP status code is returned automatically when request data validation fails?", ["handling-errors", "path-params"], r"422", "Status code 422 (Unprocessable Entity)."),
    ("Which class from fastapi.testclient is used to test FastAPI applications synchronously?", ["testing"], r"TestClient", "TestClient from fastapi.testclient."),
    ("Which underlying library powers FastAPI's TestClient?", ["testing"], r"HTTPX|httpx", "HTTPX."),
    ("Which class is used to run tasks after returning an HTTP response in FastAPI?", ["background-tasks"], r"BackgroundTasks", "BackgroundTasks."),
    ("How do you declare a path parameter in a FastAPI route decorator?", ["path-params"], r"\{item_id\}", "Using curly braces in the path string, e.g. /items/{item_id}."),
    ("What happens when a function parameter is not declared in the path string?", ["query-params"], r"query parameter", "FastAPI automatically interprets it as a query parameter."),
    ("How do you make a query parameter optional in FastAPI?", ["query-params"], r"None", "By setting its default value to None (e.g., str | None = None)."),
    ("Which BaseModel class is inherited to define request body schemas?", ["body"], r"BaseModel", "Pydantic's BaseModel."),
    ("Which function is used to declare dependencies in FastAPI endpoint parameters?", ["dependencies"], r"Depends", "Depends()."),
    ("Which OAuth2 class is used to extract a bearer token from the Authorization header?", ["security-first"], r"OAuth2PasswordBearer", "OAuth2PasswordBearer."),
    ("What is the default tokenUrl example used with OAuth2PasswordBearer in the tutorial?", ["security-first"], r"tokenUrl=\"token\"", "tokenUrl=\"token\"."),
    ("Which library is recommended in the FastAPI JWT tutorial for password hashing?", ["security-jwt"], r"pwdlib|passlib", "pwdlib (or passlib in older sections) for password hashing."),
    ("Which JWT signing algorithm is used in the FastAPI OAuth2 JWT tutorial?", ["security-jwt"], r"HS256", "HS256."),
    ("Which middleware class is used to configure Cross-Origin Resource Sharing in FastAPI?", ["cors"], r"CORSMiddleware", "CORSMiddleware."),
    ("What does allow_origins=[\"*\"] in CORSMiddleware permit?", ["cors"], r"\*", "It allows requests from all origins (domains)."),
    ("Which decorator is used to register a custom HTTP middleware function on a FastAPI app?", ["middleware"], r"@app\.middleware\(\"http\"\)", "@app.middleware(\"http\")."),
    ("What parameter name represents the next handler inside an @app.middleware(\"http\") function?", ["middleware"], r"call_next", "call_next."),
    ("Which parameter of FastAPI() replaces deprecated startup and shutdown event handlers?", ["lifespan"], r"lifespan", "The lifespan parameter using an async context manager (@asynccontextmanager)."),
    ("Which ORM library is used in the modern FastAPI SQL relational databases tutorial?", ["sql-databases"], r"SQLModel", "SQLModel."),
    ("Which command starts the FastAPI development server from the CLI?", ["first-steps"], r"fastapi dev", "fastapi dev main.py."),
    # 3 питання поза корпусом (має бути відмова)
    ("How do you configure Celery with Redis broker in FastAPI?", [], None, None),
    ("What is the default port used by PostgreSQL when running in Docker?", [], None, None),
    ("How do you deploy a FastAPI application to AWS Lambda using Mangum?", [], None, None),
]

answerable = [(q, gold, ev) for q, gold, ev, _ in BENCHMARK if gold]
print(f"Усього питань у бенчмарку: {len(BENCHMARK)} (із відповіддю: {len(answerable)}, без відповіді: {len(BENCHMARK) - len(answerable)})")

Усього питань у бенчмарку: 25 (із відповіддю: 22, без відповіді: 3)


## 1.3. Індексація

- **Модель ембедингів:** `Qwen/Qwen3-Embedding-0.6B` (розмірність 1024, максимальна довжина контексту 32 768 токенів, асиметричний промпт `query` для запитів). Обрана завдяки високій якості пошуку англомовних технічних текстів і фрагментів коду на MTEB та компактності (поміщається в памʼять GPU T4 у Google Colab разом із генератором `Qwen3-4B`).
- **Спосіб розбиття:** `RecursiveCharacterTextSplitter` з урахуванням структури Markdown (`separators=["\n## ", "\n### ", "\n\n", ". ", " ", ""]`, `keep_separator="end"`), `chunk_size=900` символів, `chunk_overlap=180` символів (перекриття 20% зберігає цілісність речень і пояснень на межах розділів). На початок кожного фрагмента додано заголовок і шлях документа (`Document: {title} ({path})`), щоб фрагменти не втрачали тематичного контексту. Отримано **199 фрагментів**.
- **Векторне сховище:** `FAISS` зі стратегією `DistanceStrategy.MAX_INNER_PRODUCT` (оскільки ембединги нормалізовані за $L_2$-нормою, скалярний добуток математично тотожний косинусній подібності). Для 199 фрагментів локальний індекс FAISS у памʼяті працює миттєво й не потребує розгортання окремого сервера векторної БД.


In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=180,
    separators=["\n## ", "\n### ", "\n\n", ". ", " ", ""],
    keep_separator="end",
)

chunks = splitter.split_documents(raw_documents)
for c in chunks:
    c.page_content = f"Document: {c.metadata['title']} ({c.metadata['path']})\n{c.page_content}"

print(f"Отримано фрагментів: {len(chunks)}")

EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": device, "model_kwargs": {"dtype": dtype}},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 16},
    query_encode_kwargs={"normalize_embeddings": True, "prompt_name": "query"},
)

vector_store = FAISS.from_documents(chunks, embeddings, distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)

def tokenize(text: str):
    return re.findall(r"\w+", text.lower())

CANDIDATES = 20
bm25 = BM25Retriever.from_documents(chunks, k=CANDIDATES, preprocess_func=tokenize)
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device=device, model_kwargs={"dtype": dtype})

Отримано фрагментів: 199


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

... (14 outputs left out)

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

## 1.4. Ретрив

Порівнюємо 4 конфігурації пошуку за метриками **hit@1**, **hit@5** та **MRR**:
1. `bm25_search` — чисто лексичний пошук;
2. `vector_search` — чисто семантичний пошук у FAISS;
3. `hybrid_search` — гібридний пошук (RRF обʼєднання векторного пошуку та BM25);
4. `rerank_search` — гібридний пошук (топ-20 кандидатів) + переранжування cross-encoder `BAAI/bge-reranker-v2-m3`.

In [4]:
def vector_search(query: str, k: int = 5):
    return vector_store.similarity_search(query, k=k)

def bm25_search(query: str, k: int = 5):
    return bm25.invoke(query)[:k]

def rrf(rankings, c: int = 60):
    scores, by_content = {}, {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            scores[doc.page_content] = scores.get(doc.page_content, 0.0) + 1.0 / (c + rank)
            by_content[doc.page_content] = doc
    return [by_content[content] for content in sorted(scores, key=scores.get, reverse=True)]

def hybrid_search(query: str, k: int = 5):
    return rrf([vector_search(query, CANDIDATES), bm25_search(query, CANDIDATES)])[:k]

def rerank_search(query: str, k: int = 5):
    candidates = hybrid_search(query, CANDIDATES)
    scores = reranker.predict([(query, doc.page_content) for doc in candidates])
    return [candidates[i] for i in np.argsort(-scores)[:k]]

def first_hit_rank(results, gold, evidence):
    for rank, doc in enumerate(results, start=1):
        if doc.metadata["doc_id"] in gold and re.search(evidence, doc.page_content, flags=re.IGNORECASE):
            return rank
    return None

def eval_retrieval(search_fn):
    ranks = [first_hit_rank(search_fn(q, 10), gold, ev) for q, gold, ev in answerable]
    hit1 = np.mean([r is not None and r <= 1 for r in ranks])
    hit5 = np.mean([r is not None and r <= 5 for r in ranks])
    mrr = np.mean([1.0 / r if r else 0.0 for r in ranks])
    return {"Метод пошуку": search_fn.__name__, "hit@1": round(float(hit1), 3), "hit@5": round(float(hit5), 3), "MRR": round(float(mrr), 3)}

retrieval_df = pd.DataFrame([eval_retrieval(fn) for fn in [bm25_search, vector_search, hybrid_search, rerank_search]])
display(retrieval_df)

,Метод пошуку,hit@1,hit@5,MRR
0,bm25_search,0.636,0.909,0.733
1,vector_search,0.727,0.909,0.797
2,hybrid_search,0.818,0.909,0.856
3,rerank_search,0.864,0.909,0.879


**Підсумок порівняння ретриверів:** кожне ускладнення пайплайну послідовно покращує точність на першій позиції (`hit@1`: `0.636 → 0.727 → 0.818 → 0.864`) та `MRR` (`0.733 → 0.797 → 0.856 → 0.879`). Водночас `hit@5 = 0.909` (`20/22`) однаковий для всіх чотирьох методів, що свідчить про 2 питання, для яких цільовий вираз-доказ не потрапив у топ-5 фрагментів (перевіряємо в розділі 1.6).

## 1.5. Генерація з цитуванням

Підключаємо генеративну модель `Qwen/Qwen3-4B-Instruct-2507` через LangChain LCEL-ланцюжок. Системний промпт вимагає відповідати виключно за знайденими фрагментами документації, ставити номер джерела у квадратних дужках (`[1]`, `[2]`) після кожного твердження та відповідати строго `«У наданих документах немає відповіді.»`, якщо інформації в контексті бракує.

In [5]:
GENERATOR_MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL_NAME)
llm = AutoModelForCausalLM.from_pretrained(GENERATOR_MODEL_NAME, dtype=dtype).to(device)

generator = pipeline(
    "text-generation",
    model=llm,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=False,
    temperature=None,
    top_p=None,
    top_k=None,
    return_full_text=False,
)
chat = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=generator), tokenizer=tokenizer)

SYSTEM_PROMPT = (
    "You are a technical assistant for FastAPI documentation. "
    "Answer ONLY using the provided documentation fragments. "
    "After each statement, cite the fragment number in square brackets, e.g. [1]. "
    "If the provided fragments do not contain the answer to the question, reply ONLY with: "
    "«У наданих документах немає відповіді.» "
    "Keep the answer concise and technically accurate."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Documentation fragments:\n\n{context}\n\nQuestion: {question}"),
])

def format_docs(docs):
    return "\n\n".join(f"[{i}] {d.page_content}" for i, d in enumerate(docs, start=1))

def build_rag_chain(search_fn, k: int = 5):
    retriever = RunnableLambda(lambda q: search_fn(q, k))
    answer_chain = (
        RunnableLambda(lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
        | prompt
        | chat
        | StrOutputParser()
    )
    return {"docs": retriever, "question": RunnablePassthrough()} | RunnablePassthrough.assign(answer=answer_chain)

rag_chain = build_rag_chain(rerank_search, k=5)

REFUSAL_PHRASE = "у наданих документах немає відповіді"
JUDGE_PROMPT = (
    "Compare the system answer with the reference answer for the given FastAPI question.\n\n"
    "Question: {question}\nReference answer: {reference}\nSystem answer: {answer}\n\n"
    "Does the system answer convey the same technical facts as the reference without contradictions? "
    "Reply with a single word: yes or no."
)

def judge_answer(question: str, reference: str, answer: str) -> bool:
    verdict = chat.invoke(JUDGE_PROMPT.format(question=question, reference=reference, answer=answer)).content
    return verdict.strip().lower().startswith("yes")

def extract_cited_ids(answer: str, docs):
    nums = [int(n) for n in re.findall(r"\[(\d+)\]", answer)]
    return [docs[n - 1].metadata["doc_id"] for n in nums if 0 < n <= len(docs)]

eval_rows = []
for q, gold, ev, ref in BENCHMARK:
    out = rag_chain.invoke(q)
    ans = out["answer"]
    found_ids = [d.metadata["doc_id"] for d in out["docs"]]
    if gold:
        correct = judge_answer(q, ref, ans)
        cited_ok = any(cid in gold for cid in extract_cited_ids(ans, out["docs"]))
    else:
        correct = REFUSAL_PHRASE in ans.lower()
        cited_ok = None
    eval_rows.append({
        "question": q,
        "correct": correct,
        "citation_ok": cited_ok,
        "gold_docs": gold,
        "retrieved_docs": found_ids,
        "answer": ans,
    })

eval_df = pd.DataFrame(eval_rows)
has_ans = eval_df["citation_ok"].notna()
print(f"Правильних відповідей (питання з відповіддю): {eval_df[has_ans]['correct'].mean():.1%}")
print(f"Коректних цитувань: {eval_df[has_ans]['citation_ok'].astype(bool).mean():.1%}")
print(f"Правильних відмов (питання без відповіді): {eval_df[~has_ans]['correct'].mean():.1%}")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (3 outputs left out)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Правильних відповідей (питання з відповіддю): 81.8%
Коректних цитувань: 95.5%
Правильних відмов (питання без відповіді): 100.0%


**Підсумок оцінювання генерації:** на 22 питаннях із відповіддю досягнуто **81.8%** правильних відповідей (18/22) та **95.5%** коректного цитування джерел (21/22), а на 3 питаннях поза корпусом — **100.0%** правильних відмов (3/3).

## 1.6. Аналіз відмов

Виводимо всі випадки, де модель відповіла неправильно, не змогла відмовитися або вказала некоректне цитування, для класифікації причин помилок.

In [6]:
failures_df = eval_df[(~eval_df["correct"]) | (eval_df["citation_ok"] == False)]
print(f"Усього проблемних випадків: {len(failures_df)} із {len(eval_df)}\n")
for idx, row in failures_df.iterrows():
    hit_doc = (not row["gold_docs"]) or any(d in row["gold_docs"] for d in row["retrieved_docs"])
    stage = "Ретрив знайшов документ (помилка генерації / судді)" if hit_doc else "Промах ретриву"
    print(f"--- Випадок #{idx + 1} ---")
    print(f"Питання:    {row['question']}")
    print(f"Очікувано:  {row['gold_docs']} | Знайдено: {row['retrieved_docs']} -> {stage}")
    print(f"Правильно:  {row['correct']} | Цитування: {row['citation_ok']}")
    print(f"Відповідь:  {row['answer']}\n")

Усього проблемних випадків: 4 із 25

--- Випадок #1 ---
Питання:    What HTTP status code does FastAPI return by default when raising HTTPException without specifying details?
Очікувано:  ['handling-errors'] | Знайдено: ['handling-errors', 'handling-errors', 'handling-errors', 'handling-errors', 'handling-errors'] -> Ретрив знайшов документ (помилка генерації / судді)
Правильно:  False | Цитування: True
Відповідь:  The documentation does not explicitly state the default HTTP status code when raising `HTTPException` without specifying details. However, it does mention that when a client requests a non-existent `item_id`, such as `"bar"`, the client receives an HTTP status code of `404` and a JSON response with `"detail": "Item not found"` [1]. This implies that a default status code of `404` is used in such cases, but this is specific to a non-existent resource scenario.

Since the question asks about the default status code when raising `HTTPException` without specifying details, and n

In [7]:
# Додаткова діагностика до 1.6:
# 1) перевірка 2 промахів ретриву (hit@5 = 0.909 -> 20/22) на rerank_search
# 2) перевірка наявності нерозгорнутих include-блоків docs_src у корпусі та 2 стрес-тести генерації
include_refs = sum(len(re.findall(r"\{\*.*?\*\}", d.page_content)) for d in raw_documents)
print(f"Нерозгорнутих include-вставок {{* ... *}} у документах: {include_refs}\n")

print("=== 1. Промахи ретриву на answerable при k=5 (rerank_search) ===")
miss_count = 0
for idx, (q, gold, ev, ref) in enumerate(BENCHMARK):
    if not gold:
        continue
    top5 = rerank_search(q, 5)
    r5 = first_hit_rank(top5, gold, ev)
    if r5 is None:
        miss_count += 1
        r10 = first_hit_rank(rerank_search(q, 10), gold, ev)
        row = eval_df.iloc[idx]
        print(f"Питання #{idx + 1}: {q}")
        print(f"  Очікувано: {gold} (regex={ev!r}) | Ранг у top-10: {r10}")
        print(f"  Знайдено top-5: {[d.metadata['doc_id'] for d in top5]}")
        print(f"  Оцінка eval_df: correct={row['correct']}, citation_ok={row['citation_ok']}")
        print(f"  Відповідь: {row['answer']}\n")
print(f"Усього промахів ретриву (hit@5): {miss_count}\n")

print("=== 2. Додаткові стрес-тести режимів відмови ===")
stress_tests = [
    ("Прогалина парсингу (код лише у docs_src)", rag_chain,
     "What exact fields and type annotations are declared in the Item class in the FastAPI Request Body tutorial?"),
    ("Лексичний промах BM25 на перефразованому запиті", build_rag_chain(bm25_search, k=5),
     "How do you make a URL query argument optional if the client omits it?"),
]
for label, chain, q in stress_tests:
    out = chain.invoke(q)
    print(f"[{label}] {q}")
    print(f"  Знайдені джерела: {[d.metadata['doc_id'] for d in out['docs']]}")
    print(f"  Відповідь: {out['answer']}\n")

Нерозгорнутих include-вставок {* ... *} у документах: 85

=== 1. Промахи ретриву на answerable при k=5 (rerank_search) ===
Питання #3: What HTTP status code is returned automatically when request data validation fails?
  Очікувано: ['handling-errors', 'path-params'] (regex='422') | Ранг у top-10: None
  Знайдено top-5: ['handling-errors', 'handling-errors', 'handling-errors', 'path-params', 'handling-errors']
  Оцінка eval_df: correct=True, citation_ok=True
  Відповідь: When request data validation fails, FastAPI automatically returns an HTTP 422 status code (Unprocessable Entity). This is because validation errors are typically classified under the 422 range, which indicates that the server understands the request but cannot process it due to semantic errors (such as invalid data types or missing fields). While the provided fragments do not explicitly state "422", they do mention that validation errors are handled with detailed error messages and that the `RequestValidationError` incl

### Розбір відмов (розділ 1.6)

Нижче наведено розбір **6 проблемних випадків** із бенчмарку (`failures_df`: `#1`, `#5`, `#7`, `#22`) та додаткової діагностики промахів ретриву/парсингу (комірка вище):

1. **Випадок #1 (`What HTTP status code does FastAPI return by default when raising HTTPException without specifying details?`)** — *помилка генерації (надмірна відмова) + неоднозначне питання й нерозгорнутий include-приклад*.  
   Усі 5 фрагментів надійшли з правильного документа `handling-errors`, але приклади Python-коду в документації FastAPI підключаються директивою `{* ../../docs_src/... *}` (у корпусі знайдено **85** таких нерозгорнутих вставок), а саме питання містить хибну передумову про «статус за замовчуванням» (`status_code` в `HTTPException` є обовʼязковим параметром). Модель спершу припустила `404` із прикладу в тексті, а в кінці відмовилася відповідати (`«У наданих документах немає відповіді.»`). **Що змінити:** розгортати `{* ... *}` файли коду під час індексації та уточнити формулювання запиту.

2. **Випадок #5 (`Which underlying library powers FastAPI's TestClient?`)** — *помилка оцінювання (помилка LLM-судді)*.  
   Система дала повністю правильну відповідь (`"FastAPI's TestClient is powered by Starlette, which in turn uses HTTPX as its underlying library for making HTTP requests. [1]"`) із коректним цитуванням `[1]` (`testing`), проте LLM-суддя (`Qwen3-4B-Instruct-2507`) відхилив її через ультрастислий еталон `"HTTPX."` та додану згадку про `Starlette`. **Що змінити:** використати сильнішу модель-суддю або додати в інструкцію судді правило не штрафувати за коректні проміжні деталі.

3. **Випадок #7 (`How do you declare a path parameter in a FastAPI route decorator?`)** — *промах ретриву на рівні фрагмента (`hit@5`) + галюцинація попри контекст + хибне цитування (`citation_ok=False`)*.  
   Діагностика показала, що фрагмент із виразом-доказом `\{item_id\}` не потрапив навіть у топ-10 (`Ранг у top-10: None`), оскільки конкретний декоратор з `{item_id}` містився у нерозгорнутому прикладі `{* docs_src/... *}`. Попри наявність `path-params` на позиції `[2]`, модель вигадала неіснуючий у FastAPI декоратор із Flask (`@app.route()`) і процитувала `[5]` (`first-steps`) замість `[2]`. **Що змінити:** підставляти код із `docs_src` у корпус та посилити системний промпт забороною підміняти синтаксис сторонніми фреймворками.

4. **Випадок #22 (`Which command starts the FastAPI development server from the CLI?`)** — *помилка оцінювання (вузький еталон / строгість судді)*.  
   Система точно вказала CLI-команду `fastapi dev [1]` і послалася на `first-steps`, але суддя поставив `False` через наявність імені файла-прикладу `main.py` в еталоні `"fastapi dev main.py."`. **Що змінити:** розширити еталон допустимими варіантами команди без привʼязки до назви файла-прикладу.

5. **Випадок #3 (`What HTTP status code is returned automatically when request data validation fails?`)** — *промах ретриву (`hit@5`) + витік знань із параметричної памʼяті попри заборону в системному промпті*.  
   Діагностична клітинка зафіксувала, що вираз-доказ `422` не потрапив навіть у топ-10 (`Ранг у top-10: None`), хоча всі 5 фрагментів були з `handling-errors` і `path-params`. Сама модель у відповіді визнала: *«While the provided fragments do not explicitly state "422"... Based strictly on the provided fragments, the exact status code is not directly stated»*, але все одно видала `**422** [1]` на основі «standard knowledge», замість того щоб відмовитися. Формально `eval_df` зарахував відповідь як `True`, хоча правило граундінгу RAG було порушено. **Що змінити:** додати код із `docs_src` до корпусу (щоб статус `422` був у фрагментах) і жорсткіше інструктувати модель відмовлятися, якщо точного факту немає в контексті.

6. **Стрес-тест прогалини парсингу (`What exact fields and type annotations are declared in the Item class in the FastAPI Request Body tutorial?`)** — *відмова через нерозгорнуті include-блоки (`docs_src`)*.  
   Усі 5 фрагментів знайдено з правильного документа `body`, однак система відповіла `«У наданих документах немає відповіді.»`, бо опис класу `Item(BaseModel)` у Markdown підключено зовнішнім блоком `{* ../../docs_src/... *}` (серед 85 подібних вставок у корпусі) і не потрапив до індексу. *(Водночас у 2-му стрес-тесті на `bm25_search` лексичний пошук помилився на 1-й позиції — повернув `testing`, але завдяки `k=5` і 4 фрагментам із `query-params` модель усе одно дала правильну відповідь `[3]`.)* **Що змінити:** реалізувати завантаження й підстановку файлів `docs_src` на етапі парсингу перед розбиттям тексту.

## 1.7. Демонстрація

Інтерактивний веб-інтерфейс на **Gradio**, у якому користувач може поставити будь-яке запитання щодо FastAPI, обрати алгоритм пошуку (`Hybrid + Reranker`, `Hybrid`, `Semantic Vector Search`), задати власний ліміт токенів генерації (`max_new_tokens` від `256` до `4096`, за замовчуванням **`2048`**) під конкретний запит і переглянути повні тексти знайдених джерел без обрізання. Нижче підготовлено 6 прикладів для всіх трьох режимів пошуку.

In [ ]:
SEARCH_MODES = {
    "Hybrid + Reranker (bge-reranker-v2-m3)": rerank_search,
    "Hybrid (FAISS + BM25 RRF)": hybrid_search,
    "Semantic Vector Search (FAISS)": vector_search,
}

def generate_demo_answer(docs, question: str, max_new_tokens: int):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Documentation fragments:\n\n{format_docs(docs)}\n\nQuestion: {question}"},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(device)
    with torch.no_grad():
        output = llm.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
        )
    gen_ids = output[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    return answer, len(gen_ids)

def ask_assistant(question: str, mode_name: str, max_tokens: int):
    if not (question or "").strip():
        return "Введіть запитання.", ""
    limit = int(max_tokens)
    docs = SEARCH_MODES[mode_name](question.strip(), 5)
    answer, used_tokens = generate_demo_answer(docs, question.strip(), limit)
    if used_tokens >= limit:
        answer += f"\n\n> ⚠️ *Досягнуто ліміту {limit} токенів. Збільште значення повзунка й повторіть запит.*"
    else:
        answer += f"\n\n_Використано токенів відповіді: {used_tokens} / {limit}_"

    sources_md = "\n\n---\n\n".join(
        f"**[{i}] {d.metadata['title']}** (`{d.metadata['path']}`)\n\n> " + d.page_content.replace("\n", "\n> ")
        for i, d in enumerate(docs, start=1)
    )
    return answer, sources_md

demo = gr.Interface(
    fn=ask_assistant,
    inputs=[
        gr.Textbox(label="Запитання до документації FastAPI", placeholder="Наприклад: How do I configure CORS in FastAPI?"),
        gr.Radio(choices=list(SEARCH_MODES.keys()), value="Hybrid + Reranker (bge-reranker-v2-m3)", label="Режим пошуку"),
        gr.Slider(minimum=256, maximum=4096, step=128, value=2048, label="Ліміт токенів відповіді (max_new_tokens)"),
    ],
    outputs=[
        gr.Markdown(label="Відповідь асистента з цитуванням"),
        gr.Markdown(label="Знайдені джерела (топ-5 фрагментів повністю)"),
    ],
    title="FastAPI Documentation RAG Assistant",
    examples=[
        ["Which class is used to run tasks after returning an HTTP response in FastAPI?", "Hybrid + Reranker (bge-reranker-v2-m3)", 2048],
        ["How do I replace deprecated startup events in FastAPI?", "Hybrid + Reranker (bge-reranker-v2-m3)", 2048],
        ["How do I configure CORS in FastAPI using CORSMiddleware?", "Hybrid (FAISS + BM25 RRF)", 2048],
        ["Which JWT signing algorithm and password hashing library are used in the OAuth2 JWT tutorial?", "Hybrid (FAISS + BM25 RRF)", 2048],
        ["How do you make a query parameter optional in FastAPI?", "Semantic Vector Search (FAISS)", 2048],
        ["How do you configure Celery with Redis broker in FastAPI?", "Semantic Vector Search (FAISS)", 2048],
    ],
)
demo.launch(share=False, inline=True)

## 1.8. Мій звіт

- **Корпус і чому потрібен RAG:** Використано 14 офіційних розділів документації фреймворку FastAPI у форматі Markdown (репозиторій `fastapi/fastapi`, ліцензія MIT, загальний обсяг після очищення — **115 197** символів). RAG потрібен для отримання точних відповідей про актуальні API (`lifespan`, `Depends`, `OAuth2PasswordBearer`, `CORSMiddleware`, `TestClient`, `SQLModel`) із посиланням на конкретний файл документації без роздування промпту всією базою знань і без потреби перенавчати модель після оновлення фреймворку.
- **Бенчмарк:** До налаштування системи складено **25 питань**: 22 питання з відповідями у корпусі (фактичні, кодові, процедурні, перефразування, охоплені всі 14 документів) та 3 питання поза корпусом (про Celery, PostgreSQL у Docker та AWS Lambda з Mangum) для тестування відмови відповідати.
- **Індексація (модель ембедингів, розбиття, сховище):** Модель ембедингів — `Qwen/Qwen3-Embedding-0.6B` (1024 виміри, асиметричний промпт `query`). Документи розбито `RecursiveCharacterTextSplitter` з урахуванням заголовків Markdown (`chunk_size=900`, `chunk_overlap=180`), на початок кожного фрагмента додано префікс `Document: {title} ({path})`. Отримано **199 фрагментів**, проіндексованих у `FAISS` (`DistanceStrategy.MAX_INNER_PRODUCT`).
- **Результати ретриву для різних конфігурацій (на 22 питаннях із відповіддю):**  
  - `bm25_search`: `hit@1` = **0.636**, `hit@5` = **0.909**, `MRR` = **0.733**;  
  - `vector_search`: `hit@1` = **0.727**, `hit@5` = **0.909**, `MRR` = **0.797**;  
  - `hybrid_search` (FAISS + BM25 RRF): `hit@1` = **0.818**, `hit@5` = **0.909**, `MRR` = **0.856**;  
  - `rerank_search` (Hybrid + `BAAI/bge-reranker-v2-m3`): `hit@1` = **0.864**, `hit@5` = **0.909**, `MRR` = **0.879**.  
  Гібридний пошук поєднав точний збіг ідентифікаторів (`BM25`) із семантичним пошуком (`+0.091` до `hit@1` відносно `vector_search`), а переранжування cross-encoder підняло `hit@1` до `0.864` і `MRR` до `0.879`. Водночас `hit@5 = 0.909` (`20/22`) вказує на 2 стабільні промахи ретриву серед перших 5 фрагментів (зокрема через те, що приклади коду в документації FastAPI підключаються директивами на кшталт `{* ../../docs_src/... *}`).
- **Генеративна модель і результати відповідей:** Використано `Qwen/Qwen3-4B-Instruct-2507` (LCEL-ланцюжок, `rerank_search` з `k=5`, жадібна генерація `do_sample=False`). На бенчмарку система досягла: правильних відповідей на питання з відповіддю — **81.8%** (`18/22`), коректних цитувань `[n]` — **95.5%** (`21/22`), правильних відмов на питання поза корпусом — **100.0%** (`3/3`).
- **Аналіз відмов:** Розбір проблемних випадків із бенчмарку (`#1`, `#5`, `#7`, `#22`) та діагностичних перевірок у розділі 1.6 виявив такі причини відмов:  
  1. *Помилка генерації (надмірна відмова) / неоднозначне питання та нерозгорнуті include-приклади* (питання `#1` про `HTTPException`): модель не побачила повної сигнатури класу у фрагменті про `404` і згенерувала відмову; рішення — розгортати директиви `docs_src` під час парсингу та точніше формулювати питання/еталон.  
  2. *Помилка оцінювання (строгість LLM-судді)* (питання `#5` про `TestClient` → `Starlette`/`HTTPX` і `#22` про CLI-команду `fastapi dev` без `main.py`): фактичні відповіді й цитування `[1]` були правильними, але суддя на базі тієї самої 4B-моделі відхилив їх через ультрастислі еталони; рішення — сильніша модель-суддя або розгорнутіші еталони.  
  3. *Галюцинація попри контекст + хибне цитування* (питання `#7` про параметр шляху): модель змішала FastAPI з Flask (`@app.route()`) і процитувала `[5]` (`first-steps`) замість `[2]` (`path-params`); рішення — включення кодових прикладів у корпус та інструкція не підміняти синтаксис сторонніми фреймворками.  
  4. *Промах ретриву (`hit@5 = 0.909`, 2 із 22 питань) та нерозгорнуті `docs_src`-блоки (85 вставок `{* ... *}`)*: діагностика в розділі 1.6 підтвердила, що докази для питань `#3` (`422`) і `#7` (`{item_id}`) не потрапили навіть у топ-10; на питанні `#3` модель порушила обмеження контексту й підставила `422` зі «standard knowledge», а на запиті про поля класу `Item` у `body` система змушена була відмовити через відсутність коду `docs_src` в індексі.
- **Висновок:** Побудований RAG-пайплайн із гібридним пошуком та переранжуванням `bge-reranker-v2-m3` забезпечує високу точність ранжування (`MRR = 0.879`, `hit@1 = 0.864`), надійне цитування (`95.5%`) та безпомилкову відмову на запитах поза корпусом (`100.0%`). Головний резерв покращення якості лежить не у зміні ретривера, а на етапі підготовки даних (розгортання зовнішніх сніпетів коду `docs_src`) та використанні потужнішого LLM-судді.
